**Import packages and dependecies**

In [ ]:
%pip install -q -e ..

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
import kagglehub

from tda.rips import VietorisRips
from tda.bottleneck import bottleneck_distance

### 1. Load in data inputs

The dataset shown below is from the Digital Payment Fraud Detection Benchmark which contains simulated, large-scale digital payment transaction data. The data spans one full calendar year with a known feature drift occurring mid-year. We will use this data to see how TDA can be applied to detect this feature drift

**Months 1-6**
- Merchant-driven fraud influence
- Risk primarily influenced by merchant and IP-level signals

**Months 7-12**
- Increased velocity-based fraud
- Low-amount micro-transactions
- Higher sensitivity to international activity

For further information about the data, see [Digital Payment Fraud Detection Benchmark](https://www.kaggle.com/datasets/rohit8527kmr7518/digital-payment-fraud-detection-benchmark?select=transactions_train.csv).

In [ ]:
# Download data fram Kaggle
path = kagglehub.dataset_download("rohit8527kmr7518/digital-payment-fraud-detection-benchmark")

# Train contains all transactions for months 1-6
transactions_train_df = pl.read_csv(f"{path}/transactions_train.csv")
# Test contains all transactions for months 7-12
transactions_test_df = pl.read_csv(f"{path}/transactions_test.csv")

# Show transactions
transactions_train_df.limit(50).show()

Limit to features fields (remove any primary keys or identifier fields).

In [ ]:
primary_keys_list = [
    "transaction_id",
    "transaction_time",
    "customer_id",
    "merchant_id",
]

target_outcomes_list = [
    "is_fraud",
    "post_auth_risk_score",
]

features_list = [
    "account_age_days",
    "credit_score_band",
    "kyc_level",
    "avg_monthly_spend",
    "merchant_risk_score",
    "transaction_amount",
    "payment_channel",
    "device_type",
    "is_international",
    "ip_risk_score",
    "txn_count_1h",
    "txn_count_24h",
    "failed_txn_count_24h",
    "geo_distance_from_last_txn",
    "amount_deviation_from_user_mean",
]

preprocessed_train_df = transactions_train_df.select(features_list)
preprocessed_test_df = transactions_test_df.select(features_list)

print(f"Total # of features: {len(features_list)}")

Remove columns if they have high missingness rate.

In [ ]:
combined_df = pl.concat([preprocessed_train_df, preprocessed_test_df])

# Initialize parameters
columns_list = combined_df.columns
tot_n_rows = combined_df.shape[0]
missing_threshold = 0.4 # Columns w/ missing rate less than or equal to threshold are kept

# Calculate the percent of missing in each column
missing_count_df = combined_df.null_count()/tot_n_rows
missing_count_df = missing_count_df.unpivot(
    on=columns_list,
    variable_name="Variable Name",
    value_name="p_missing"
)

# Remove columns from data
columns_failed_threshold_list = missing_count_df.filter(pl.col("p_missing") > missing_threshold).select("Variable Name").to_series()
preprocessed_train_df = preprocessed_train_df.select([c for c in columns_list if c not in columns_failed_threshold_list])
preprocessed_test_df = preprocessed_test_df.select([c for c in columns_list if c not in columns_failed_threshold_list])

print(f"Total # of columns removed due to high missing rate: {len(columns_failed_threshold_list)}")

### 2. Create persistence diagrams